# 개별종목 조합I — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5050,0.5012,0.0038,0.2853,0.3536,0.0759,0.3821,0.0454,0.1090
1,2,balanced,980,20150123,20150421,0.4029,0.3978,0.0050,0.3558,0.3718,0.0709,0.3862,0.2008,0.2921
2,3,balanced,1210,20151228,20160328,0.3734,0.3762,-0.0027,0.3702,0.3700,0.0576,0.3770,0.3322,0.3576
3,4,balanced,1439,20161202,20170228,0.4656,0.4617,0.0039,0.3562,0.3810,0.1005,0.4112,0.1495,0.2577
4,5,balanced,1669,20171113,20180207,0.4254,0.3901,0.0353,0.3851,0.3997,0.1133,0.4066,0.2575,0.3397
5,6,balanced,1899,20181024,20190118,0.4153,0.3725,0.0429,0.4151,0.4205,0.1335,0.4176,0.4704,0.4321
6,7,balanced,2129,20190930,20191224,0.4703,0.4781,-0.0079,0.3381,0.3712,0.0860,0.4116,0.1777,0.2801
7,8,balanced,2359,20200902,20201130,0.4126,0.3476,0.0650,0.4123,0.4182,0.1254,0.4130,0.4839,0.4338
8,9,balanced,2589,20210806,20211105,0.3920,0.3916,0.0003,0.3737,0.3835,0.0735,0.3846,0.2744,0.3382
9,10,balanced,2818,20220714,20221012,0.3444,0.3454,-0.0010,0.3409,0.3455,0.0204,0.3685,0.2640,0.3117


,OOS 폴드 평균
accuracy,0.4178
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0209
macro_f1,0.3685
balanced_accuracy,0.3843
mcc,0.0884
pr_auc_macro_ovr,0.3974
down_recall,0.2730
core_harmonic_mean,0.3233


재실행 명령: python scripts/run_stock_model_experiment.py
